# BDD100K: Learn by Doing — All 8 Tasks in One Notebook

**Paper:** BDD100K: A Diverse Driving Dataset for Heterogeneous Multitask Learning (Yu et al., 2020)

**Strategy:** Use pretrained models — no training needed. Each task takes ~5 minutes.

| # | Task | Model Used | Time |
|---|------|-----------|------|
| 1 | Image Tagging | CLIP (zero-shot) | ~5 min |
| 2 | Object Detection | YOLOv8 | ~5 min |
| 3 | Drivable Area | SegFormer (road classes) | ~5 min |
| 4 | Lane Marking | Hough + CLRNet | ~10 min |
| 5 | Semantic Segmentation | SegFormer-B5-Cityscapes | ~5 min |
| 6 | Multitask Visualization | Stack above outputs | ~5 min |
| 7 | Multi-Object Tracking | YOLOv8 + ByteTrack | ~10 min |
| 8 | MOTS | YOLOv8-seg + ByteTrack | ~10 min |

> **Runtime:** Select `Runtime > Change runtime type > T4 GPU` (or A100 with Colab Pro)

## Step 0A: Install Dependencies

In [ ]:
%%capture
!pip install ultralytics transformers supervision opencv-python-headless
!pip install bdd100k  # official BDD100K toolkit

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

## Step 0B: Download Sample Data

Two options — pick one:

**Option A (Kaggle):** Requires `kaggle.json` (from kaggle.com > Account > Create API Token)  
**Option B (Upload):** Upload any driving images/video manually

In [ ]:
import os

# ─── OPTION A: Kaggle download ────────────────────────────────────────────────
# Upload your kaggle.json when prompted, then run this cell

USE_KAGGLE = True  # Set False to use Option B

if USE_KAGGLE:
    from google.colab import files
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    # Download a small BDD100K sample (100 images + 1 video)
    os.makedirs('/content/bdd100k', exist_ok=True)
    !kaggle datasets download -d marquis03/bdd100k -p /content/bdd100k --unzip -q
    
    # Use just 20 images for speed
    import glob
    all_imgs = glob.glob('/content/bdd100k/**/*.jpg', recursive=True)[:20]
    print(f"Found {len(all_imgs)} images")
    IMAGES = all_imgs
    VIDEO = glob.glob('/content/bdd100k/**/*.mov', recursive=True)[:1] or \
            glob.glob('/content/bdd100k/**/*.mp4', recursive=True)[:1]
    VIDEO = VIDEO[0] if VIDEO else None

In [ ]:
# ─── OPTION B: Manual upload ─────────────────────────────────────────────────
# Upload 5-20 driving images (JPG/PNG) and optionally 1 video (MP4)
# Skip this cell if you used Option A

if not USE_KAGGLE:
    from google.colab import files
    print("Upload driving images (JPG/PNG) and optionally a video (MP4):")
    uploaded = files.upload()

    os.makedirs('/content/bdd100k/images', exist_ok=True)
    IMAGES, VIDEO = [], None
    for name, data in uploaded.items():
        path = f'/content/bdd100k/images/{name}'
        with open(path, 'wb') as f:
            f.write(data)
        if name.lower().endswith(('.jpg', '.jpeg', '.png')):
            IMAGES.append(path)
        elif name.lower().endswith('.mp4'):
            VIDEO = path
    print(f"Loaded {len(IMAGES)} images, video={VIDEO}")

In [ ]:
# Preview sample images
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

n_preview = min(6, len(IMAGES))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    if i < n_preview:
        img = mpimg.imread(IMAGES[i])
        ax.imshow(img)
        ax.set_title(os.path.basename(IMAGES[i])[:30])
    ax.axis('off')
plt.suptitle('Sample BDD100K Images', fontsize=14)
plt.tight_layout()
plt.show()

---
## Task 1: Image Tagging

**Paper:** Section 3.1 — Image-level labels for weather (6), scene (6), time-of-day (3)  
**Approach:** CLIP zero-shot classification — no finetuning needed  
**Key insight:** BDD100K has ~50% day, ~50% night images to reduce domain bias

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model = clip_model.to(device)

# BDD100K label sets (from paper)
WEATHER = ["clear weather", "overcast sky", "rainy weather", "snowy weather",
           "foggy weather", "partly cloudy sky"]
SCENE   = ["city street", "highway", "residential area", "parking lot",
           "gas station", "tunnel"]
TIMEDAY = ["daytime driving", "night driving", "dawn or dusk driving"]

def classify_image(img_path):
    img = Image.open(img_path).convert('RGB')
    results = {}
    for label_set_name, labels in [("Weather", WEATHER), ("Scene", SCENE), ("Time", TIMEDAY)]:
        inputs = clip_proc(text=labels, images=img, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = clip_model(**inputs).logits_per_image[0]
            probs  = logits.softmax(dim=-1).cpu().numpy()
        results[label_set_name] = list(zip(labels, probs))
    return img, results

# Run on first 4 images
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for row, img_path in enumerate(IMAGES[:4]):
    img, results = classify_image(img_path)
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"Image {row+1}", fontsize=10)
    axes[row, 0].axis('off')
    for col, (cat, preds) in enumerate(results.items(), 1):
        labels_  = [p[0].replace(' weather','').replace(' driving','').replace(' area','') for p in preds]
        scores   = [p[1] for p in preds]
        colors   = ['#2196F3' if s == max(scores) else '#E0E0E0' for s in scores]
        axes[row, col].barh(labels_, scores, color=colors)
        top = labels_[scores.index(max(scores))]
        axes[row, col].set_title(f"{cat}: {top}", fontsize=9)
        axes[row, col].set_xlim(0, 1)
        axes[row, col].tick_params(labelsize=8)

plt.suptitle('Task 1: Image Tagging (CLIP Zero-Shot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Paper baseline (DLA-34): ~77% weather, ~86% scene accuracy. CLIP is zero-shot!")

---
## Task 2: Object Detection

**Paper:** Section 3.2 — 10 categories, 1M+ bounding boxes, long-tail distribution  
**Approach:** YOLOv8 pretrained on COCO (same 10 categories overlap with BDD100K)  
**Key insight:** ~50% of instances are occluded (Fig. 3b) — see how the model handles them

In [ ]:
from ultralytics import YOLO
import supervision as sv
import cv2
import numpy as np
import matplotlib.pyplot as plt

# BDD100K's 10 detection categories
BDD_CLASSES = ['person', 'rider', 'car', 'truck', 'bus', 'train',
               'motorcycle', 'bicycle', 'traffic light', 'traffic sign']

det_model = YOLO('yolov8x.pt')  # x = largest, best accuracy

# Run on 4 images
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
category_counts = {c: 0 for c in BDD_CLASSES}

for i, (img_path, ax) in enumerate(zip(IMAGES[:4], axes.flat)):
    results = det_model(img_path, conf=0.35, verbose=False)[0]
    annotated = results.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)

    # Count BDD100K-relevant detections
    n_det = len(results.boxes)
    classes_det = [det_model.names[int(c)] for c in results.boxes.cls]
    for c in classes_det:
        if c in category_counts:
            category_counts[c] += 1
    ax.set_title(f"Image {i+1}: {n_det} objects detected", fontsize=11)
    ax.axis('off')

plt.suptitle('Task 2: Object Detection (YOLOv8x)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show category distribution (mirrors paper Fig. 3a)
active = {k: v for k, v in category_counts.items() if v > 0}
if active:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(active.keys(), active.values(), color='#2196F3')
    ax.set_title('Category Distribution (your sample vs. paper Fig. 3a: Car >> Sign >> Light >> Person)')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

---
## Task 3: Drivable Area Segmentation

**Paper:** Section 3.4 — Two classes: *directly drivable* (red) and *alternatively drivable* (blue)  
**Approach:** SegFormer-B5 (Cityscapes) — extract road/sidewalk classes as proxy  
**Key insight:** On highways, drivable area ≈ lane boundaries. In residential areas, it's context-dependent.

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

seg_processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
seg_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b5-finetuned-cityscapes-1024-1024").to(device)
seg_model.eval()

# Cityscapes label IDs → BDD100K drivable area mapping
# ID 0=road (directly drivable), ID 1=sidewalk (alternative), others=background
ROAD_ID = 0
SIDEWALK_ID = 1

def predict_drivable(img_path):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    inputs = seg_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = seg_model(**inputs).logits  # [1, 19, H/4, W/4]
    upsampled = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
    pred = upsampled.argmax(dim=1)[0].cpu().numpy()  # [H, W]

    # Create drivable area overlay (red=direct, blue=alternative)
    overlay = np.array(img)
    direct_mask = (pred == ROAD_ID)
    alt_mask    = (pred == SIDEWALK_ID)
    overlay[direct_mask] = overlay[direct_mask] * 0.4 + np.array([220, 50, 50]) * 0.6
    overlay[alt_mask]    = overlay[alt_mask]    * 0.4 + np.array([50, 50, 220]) * 0.6
    return img, overlay.astype(np.uint8), direct_mask, alt_mask

fig, axes = plt.subplots(len(IMAGES[:4]), 3, figsize=(18, 5 * len(IMAGES[:4])))
if len(IMAGES[:4]) == 1:
    axes = axes[np.newaxis, :]

for i, img_path in enumerate(IMAGES[:4]):
    img, overlay, dm, am = predict_drivable(img_path)
    axes[i, 0].imshow(img);      axes[i, 0].set_title('Original');             axes[i, 0].axis('off')
    axes[i, 1].imshow(overlay);  axes[i, 1].set_title('Drivable Area');        axes[i, 1].axis('off')
    # Show binary masks
    combined = np.zeros((*dm.shape, 3), dtype=np.uint8)
    combined[dm] = [220, 50, 50]   # red = direct
    combined[am] = [50, 50, 220]   # blue = alternative
    axes[i, 2].imshow(combined); axes[i, 2].set_title('Mask (red=direct, blue=alt)'); axes[i, 2].axis('off')
    print(f"Image {i+1}: direct={dm.sum():,}px ({dm.mean()*100:.1f}%), alt={am.sum():,}px ({am.mean()*100:.1f}%)")

plt.suptitle('Task 3: Drivable Area Segmentation (SegFormer-B5 Cityscapes)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 4: Lane Marking Detection

**Paper:** Section 3.3 — 8 marking types, attributes: direction (parallel/perpendicular) + continuity  
**Approach Part A:** Classic Hough transform baseline  
**Approach Part B:** CLRNet deep learning model  
**Key insight:** BDD100K has 100K lane sequences vs. Caltech's 1,224 images — 80x more data

In [ ]:
# Part A: Classic Hough Transform baseline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def detect_lanes_hough(img_path):
    img = cv2.imread(img_path)
    H, W = img.shape[:2]

    # Focus on bottom 60% of image (where lanes appear)
    roi_top = int(H * 0.4)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    # Mask to ROI trapezoid
    mask = np.zeros_like(edges)
    poly = np.array([[
        (int(W * 0.1), H), (int(W * 0.45), roi_top),
        (int(W * 0.55), roi_top), (int(W * 0.9), H)
    ]])
    cv2.fillPoly(mask, poly, 255)
    masked_edges = cv2.bitwise_and(edges, mask)

    # Hough lines
    lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180,
                             threshold=50, minLineLength=80, maxLineGap=30)
    annotated = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(annotated, (x1, y1), (x2, y2), (0, 255, 0), 3)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), masked_edges, annotated, len(lines) if lines is not None else 0

fig, axes = plt.subplots(len(IMAGES[:4]), 3, figsize=(18, 5 * len(IMAGES[:4])))
if len(IMAGES[:4]) == 1:
    axes = axes[np.newaxis, :]

for i, img_path in enumerate(IMAGES[:4]):
    orig, edges, annotated, n_lines = detect_lanes_hough(img_path)
    axes[i, 0].imshow(orig);      axes[i, 0].set_title('Original');        axes[i, 0].axis('off')
    axes[i, 1].imshow(edges, cmap='gray'); axes[i, 1].set_title('Canny Edges (ROI)'); axes[i, 1].axis('off')
    axes[i, 2].imshow(annotated); axes[i, 2].set_title(f'Hough Lines ({n_lines})'); axes[i, 2].axis('off')

plt.suptitle('Task 4A: Lane Marking — Hough Transform Baseline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Limitation: Hough fails on curves, occlusion, faded markings.")
print("BDD100K paper uses deep learning with 8 lane categories + direction/continuity attributes.")

In [ ]:
# Part B: CLRNet — Deep Learning Lane Detection
# CLRNet uses cross-layer refinement on BDD100K / CULane / TuSimple
%%capture
!pip install clrnet  # or: pip install git+https://github.com/Turoad/CLRNet

# Note: CLRNet requires model weights download (~100MB)
# If install fails, the Hough baseline above already demonstrates the concept.
# The key paper finding: BDD100K lane annotations = 100K sequences vs Caltech's 1.2K images

In [ ]:
# CLRNet inference (run only if clrnet installed successfully)
try:
    import clrnet
    from clrnet.utils.net_utils import load_network
    print("CLRNet available — running deep lane detection...")
    # Full CLRNet usage requires config + weights — see: github.com/Turoad/CLRNet
    # For now, print the key paper comparison:
except ImportError:
    print("CLRNet not installed. Using Hough baseline is fine for learning.")
    print()
    print("Paper Table 1 — Lane Dataset Comparison:")
    print(f"{'Dataset':<25} {'Training':>10} {'Total':>10} {'Sequences':>12}")
    print("-" * 60)
    print(f"{'Caltech Lanes':<25} {'—':>10} {'1,224':>10} {'4':>12}")
    print(f"{'Road Marking [Wu]':<25} {'—':>10} {'1,443':>10} {'29':>12}")
    print(f"{'KITTI-ROAD':<25} {'289':>10} {'579':>10} {'—':>12}")
    print(f"{'VPGNet':<25} {'14,783':>10} {'21,097':>10} {'—':>12}")
    print(f"{'BDD100K':<25} {'70,000':>10} {'100,000':>10} {'100,000':>12}  ← 80x more!")

---
## Task 5: Semantic Segmentation

**Paper:** Section 3.5 — 40 object classes, pixel-level labels, 10K images  
**Approach:** SegFormer-B5 finetuned on Cityscapes (19 classes — closest public model)  
**Key insight from paper:** Model trained on Cityscapes FAILS on BDD100K (Fig. 9) — domain gap!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import torch.nn.functional as F

# Cityscapes 19-class colormap
CITYSCAPES_PALETTE = np.array([
    [128, 64,128], [244, 35,232], [ 70, 70, 70], [102,102,156],
    [190,153,153], [153,153,153], [250,170, 30], [220,220,  0],
    [107,142, 35], [152,251,152], [ 70,130,180], [220, 20, 60],
    [255,  0,  0], [  0,  0,142], [  0,  0, 70], [  0, 60,100],
    [  0, 80,100], [  0,  0,230], [119, 11, 32]
], dtype=np.uint8)

CITYSCAPES_LABELS = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain', 'sky',
    'person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle'
]

def segment_image(img_path):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    inputs = seg_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = seg_model(**inputs).logits
    pred = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
    pred = pred.argmax(dim=1)[0].cpu().numpy()

    # Colorize
    colored = CITYSCAPES_PALETTE[pred.clip(0, 18)]
    blend = (np.array(img) * 0.5 + colored * 0.5).astype(np.uint8)

    # Class distribution
    unique, counts = np.unique(pred, return_counts=True)
    dist = {CITYSCAPES_LABELS[u]: c / pred.size for u, c in zip(unique, counts) if u < 19}
    return img, blend, pred, dist

fig, axes = plt.subplots(len(IMAGES[:4]), 3, figsize=(18, 5 * len(IMAGES[:4])))
if len(IMAGES[:4]) == 1:
    axes = axes[np.newaxis, :]

for i, img_path in enumerate(IMAGES[:4]):
    img, blend, pred, dist = segment_image(img_path)
    axes[i, 0].imshow(img);    axes[i, 0].set_title('Original');                axes[i, 0].axis('off')
    axes[i, 1].imshow(blend);  axes[i, 1].set_title('Semantic Segmentation');   axes[i, 1].axis('off')

    # Class distribution bar
    top_classes = sorted(dist.items(), key=lambda x: -x[1])[:8]
    names, vals = zip(*top_classes)
    colors_bar = [CITYSCAPES_PALETTE[CITYSCAPES_LABELS.index(n)] / 255. for n in names]
    axes[i, 2].barh(names, [v*100 for v in vals], color=colors_bar)
    axes[i, 2].set_xlabel('% of pixels')
    axes[i, 2].set_title('Class Distribution')

plt.suptitle('Task 5: Semantic Segmentation (SegFormer-B5, Cityscapes weights)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Paper finding: mIoU drops ~5pts when Cityscapes model is tested on BDD100K (domain gap!)")
print("Key failure: traffic signs look different in US vs Germany.")

---
## Task 6: Multitask Visualization

**Paper:** Section 5 — Heterogeneous multitask learning: combining all tasks on one backbone  
**Approach:** Stack Tasks 1-5 outputs on same image in a single dashboard  
**Key insight from paper:** Joint training detection+segmentation improves mIoU from 56.9→58.3 (Table 8)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image
import torch
import torch.nn.functional as F

def full_pipeline(img_path):
    """Run all 5 tasks on one image — simulates the BDD100K multitask model."""

    # Task 1: Tagging
    img_pil = Image.open(img_path).convert('RGB')
    _, tag_results = classify_image(img_path)
    weather_top = max(tag_results['Weather'], key=lambda x: x[1])[0]
    scene_top   = max(tag_results['Scene'],   key=lambda x: x[1])[0]
    time_top    = max(tag_results['Time'],     key=lambda x: x[1])[0]

    # Task 2: Detection
    det_results = det_model(img_path, conf=0.35, verbose=False)[0]
    det_img = cv2.cvtColor(det_results.plot(), cv2.COLOR_BGR2RGB)

    # Task 3+5: Drivable + Semantic (same SegFormer pass)
    W, H = img_pil.size
    inputs = seg_processor(images=img_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = seg_model(**inputs).logits
    upsampled = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
    pred = upsampled.argmax(dim=1)[0].cpu().numpy()
    seg_colored = CITYSCAPES_PALETTE[pred.clip(0, 18)]
    seg_blend = (np.array(img_pil) * 0.4 + seg_colored * 0.6).astype(np.uint8)

    # Task 3: Drivable overlay on detection image
    multitask = det_img.copy()
    road_mask = (pred == 0)
    alt_mask  = (pred == 1)
    multitask[road_mask] = (multitask[road_mask] * 0.5 + np.array([220,50,50]) * 0.5).astype(np.uint8)
    multitask[alt_mask]  = (multitask[alt_mask]  * 0.5 + np.array([50,50,220]) * 0.5).astype(np.uint8)

    # Task 4: Lane lines on top
    gray  = cv2.cvtColor(cv2.cvtColor(multitask, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
    poly  = np.array([[(int(W*.1),H),(int(W*.45),int(H*.4)),(int(W*.55),int(H*.4)),(int(W*.9),H)]])
    mask_h = np.zeros_like(edges); cv2.fillPoly(mask_h, poly, 255)
    lines = cv2.HoughLinesP(cv2.bitwise_and(edges, mask_h), 1, np.pi/180, 50, 80, 30)
    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            cv2.line(multitask, (x1,y1), (x2,y2), (0,255,0), 2)

    return img_pil, det_img, seg_blend, multitask, weather_top, scene_top, time_top

# Run on one image
img_pil, det_img, seg_blend, multitask, weather, scene, timeday = full_pipeline(IMAGES[0])

fig = plt.figure(figsize=(20, 10))
gs = GridSpec(2, 4, figure=fig)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(img_pil);    ax0.set_title('Original');                    ax0.axis('off')
ax1 = fig.add_subplot(gs[0, 1]); ax1.imshow(det_img);    ax1.set_title('Task 2: Detection');           ax1.axis('off')
ax2 = fig.add_subplot(gs[0, 2]); ax2.imshow(seg_blend);  ax2.set_title('Task 5: Semantic Seg');        ax2.axis('off')
ax3 = fig.add_subplot(gs[:, 3]); ax3.imshow(multitask);  ax3.set_title('Tasks 2+3+4: Multitask');      ax3.axis('off')

# Task 1 text box
ax4 = fig.add_subplot(gs[1, 0:3])
ax4.axis('off')
info = (f"Task 1 — Image Tagging:\n"
        f"  Weather: {weather}\n"
        f"  Scene:   {scene}\n"
        f"  Time:    {timeday}\n\n"
        f"Paper insight: Special training strategies needed for heterogeneous multitask models.\n"
        f"Joint detection+seg: mIoU 56.9 → 58.3 (Table 8). MOT+detection: MOTA 55.0 → 56.7 (Table 7).")
ax4.text(0.05, 0.5, info, transform=ax4.transAxes, fontsize=12,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Task 6: Multitask Dashboard — All Tasks on One Image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 7: Multi-Object Tracking (MOT)

**Paper:** Section 3.6 — 2,000 videos, 130K track identities, 3.3M bounding boxes  
**Approach:** YOLOv8 + ByteTrack (built-in to ultralytics)  
**Key insight from paper:** BDD100K MOT = 10x more sequences than MOT17, half instances are occluded

> Requires a video clip. If you don't have one, download a short dashcam clip from YouTube.

In [ ]:
# Optional: download a short dashcam video for tracking demo
# Uncomment and replace URL with a direct mp4 link you have permission to use
# !wget -q -O /content/demo.mp4 "YOUR_VIDEO_URL_HERE"
# VIDEO = '/content/demo.mp4'

# Check if we have a video
if VIDEO is None:
    print("No video found. Please upload an MP4 or provide a URL above.")
    print("Alternatively, we'll create a synthetic video from your images.")

    # Create a slideshow video from images for demo
    import cv2
    h, w = 720, 1280
    VIDEO = '/content/demo_slideshow.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(VIDEO, fourcc, 3.0, (w, h))  # 3 fps
    for img_path in IMAGES[:min(10, len(IMAGES))]:
        frame = cv2.imread(img_path)
        frame = cv2.resize(frame, (w, h))
        for _ in range(10):  # hold each frame for 10 frames = ~3s
            out.write(frame)
    out.release()
    print(f"Created synthetic slideshow: {VIDEO}")

print(f"Using video: {VIDEO}")

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
import numpy as np

mot_model = YOLO('yolov8x.pt')

# Track objects through video using ByteTrack
track_history = defaultdict(list)
frames_with_tracks = []
max_frames = 60  # process first 60 frames = ~2 seconds

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {fps:.1f} fps, {total_frames} frames total, processing first {max_frames}")

frame_idx = 0
while cap.isOpened() and frame_idx < max_frames:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLOv8 tracking with ByteTrack
    results = mot_model.track(frame, persist=True, tracker="bytetrack.yaml",
                               conf=0.3, verbose=False)[0]
    annotated = results.plot()

    # Draw track trails
    if results.boxes.id is not None:
        boxes = results.boxes.xywh.cpu().numpy()
        track_ids = results.boxes.id.int().cpu().numpy()
        for box, tid in zip(boxes, track_ids):
            cx, cy = int(box[0]), int(box[1])
            track_history[tid].append((cx, cy))
            if len(track_history[tid]) > 30:
                track_history[tid].pop(0)
            # Draw trail
            pts = np.array(track_history[tid], dtype=np.int32)
            if len(pts) > 1:
                cv2.polylines(annotated, [pts], False, (0, 255, 255), 2)

    if frame_idx % 10 == 0:
        frames_with_tracks.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    frame_idx += 1

cap.release()
print(f"Tracked {len(track_history)} unique objects across {frame_idx} frames")

# Show sample tracked frames
n_show = min(6, len(frames_with_tracks))
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for i, ax in enumerate(axes.flat):
    if i < n_show:
        ax.imshow(frames_with_tracks[i])
        ax.set_title(f'Frame {i*10}')
    ax.axis('off')

plt.suptitle(f'Task 7: Multi-Object Tracking (YOLOv8 + ByteTrack)\n'
             f'Found {len(track_history)} unique track IDs | '
             f'Paper: BDD100K has 130K track identities vs MOT17\'s 1,638',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Track statistics — mirrors paper Fig. 7 (track length, box size distributions)
track_lengths = {tid: len(pts) for tid, pts in track_history.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Track length distribution
lengths = list(track_lengths.values())
axes[0].hist(lengths, bins=20, color='#2196F3', edgecolor='black')
axes[0].set_xlabel('Track length (frames)')
axes[0].set_ylabel('Count')
axes[0].set_title('Track Length Distribution\n(Paper Fig. 7: BDD100K has 100s of frames per track)')

# Occlusion simulation — tracks that disappeared and reappeared
axes[1].bar(['Short (<5)', 'Medium (5-20)', 'Long (>20)'],
            [sum(1 for l in lengths if l < 5),
             sum(1 for l in lengths if 5 <= l <= 20),
             sum(1 for l in lengths if l > 20)],
            color=['#F44336','#FF9800','#4CAF50'])
axes[1].set_title('Track Length Categories\n(Paper: 49,418 occlusion events in full dataset)')
axes[1].set_ylabel('Count')

plt.suptitle('Task 7: MOT Statistics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 8: Multi-Object Tracking and Segmentation (MOTS)

**Paper:** Section 3.7 — 90 videos, 60 train / 10 val / 20 test, 6.3K track identities  
**Approach:** YOLOv8-seg + ByteTrack = track AND segment simultaneously  
**Key insight from paper (Table 9):** Fine-tuning from instance-seg pre-training: MOTSA 30.4→33.7

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

mots_model = YOLO('yolov8x-seg.pt')  # segmentation model

# Assign consistent colors per track ID
def get_track_color(track_id):
    rng = random.Random(track_id)
    return [rng.randint(50, 255), rng.randint(50, 255), rng.randint(50, 255)]

mots_frames = []
mots_track_history = defaultdict(list)
max_frames = 30

cap = cv2.VideoCapture(VIDEO)
frame_idx = 0

while cap.isOpened() and frame_idx < max_frames:
    ret, frame = cap.read()
    if not ret:
        break

    results = mots_model.track(frame, persist=True, tracker="bytetrack.yaml",
                                conf=0.3, verbose=False)[0]
    annotated = frame.copy()

    if results.boxes.id is not None and results.masks is not None:
        track_ids = results.boxes.id.int().cpu().numpy()
        masks = results.masks.data.cpu().numpy()
        boxes = results.boxes.xywh.cpu().numpy()

        for tid, mask, box in zip(track_ids, masks, boxes):
            color = get_track_color(int(tid))
            H, W = frame.shape[:2]
            mask_resized = cv2.resize(mask, (W, H)) > 0.5

            # Color fill the instance mask
            overlay = annotated.copy()
            overlay[mask_resized] = color
            annotated = cv2.addWeighted(annotated, 0.5, overlay, 0.5, 0)

            # Draw track ID
            cx, cy = int(box[0]), int(box[1])
            cv2.putText(annotated, f'ID:{tid}', (cx-20, cy-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Track trail
            mots_track_history[tid].append((cx, cy))
            if len(mots_track_history[tid]) > 20:
                mots_track_history[tid].pop(0)
            pts = np.array(mots_track_history[tid], dtype=np.int32)
            if len(pts) > 1:
                cv2.polylines(annotated, [pts], False, color, 2)

    if frame_idx % 5 == 0:
        mots_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    frame_idx += 1

cap.release()
print(f"MOTS: tracked {len(mots_track_history)} unique instances with segmentation masks")

n_show = min(6, len(mots_frames))
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for i, ax in enumerate(axes.flat):
    if i < n_show:
        ax.imshow(mots_frames[i])
        ax.set_title(f'Frame {i*5}')
    ax.axis('off')

plt.suptitle(
    f'Task 8: MOTS — Multi-Object Tracking + Segmentation (YOLOv8x-seg + ByteTrack)\n'
    f'Each color = unique tracked instance | '
    f'Paper Table 9: Det+T+I+S achieves MOTSA=23.3, AP=41.4',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bonus: Domain Gap Experiment

**Paper Section 4 + Fig. 9:** Model trained on Cityscapes fails on BDD100K.  
Reproduce the key finding by comparing predictions on a city vs. highway image.

In [ ]:
# Pick 2 images: one city-like, one highway-like (or just use first 2)
img1_path = IMAGES[0]
img2_path = IMAGES[1] if len(IMAGES) > 1 else IMAGES[0]

_, blend1, _, dist1 = segment_image(img1_path)
_, blend2, _, dist2 = segment_image(img2_path)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(blend1)
top1 = sorted(dist1.items(), key=lambda x: -x[1])[:3]
axes[0].set_title(f'Image 1\nTop classes: {[f"{k}:{v*100:.0f}%" for k,v in top1]}')
axes[0].axis('off')

axes[1].imshow(blend2)
top2 = sorted(dist2.items(), key=lambda x: -x[1])[:3]
axes[1].set_title(f'Image 2\nTop classes: {[f"{k}:{v*100:.0f}%" for k,v in top2]}')
axes[1].axis('off')

plt.suptitle(
    'Bonus: Domain Gap Experiment\n'
    'Paper finding: DRN trained on Cityscapes confuses US traffic signs as sky/vegetation\n'
    'This is why BDD100K exists — to train models that generalize beyond German city streets',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary: What You've Learned

| Task | Paper Section | Key Finding |
|------|--------------|-------------|
| 1. Image Tagging | 3.1 | 50/50 day/night split reduces temporal bias |
| 2. Object Detection | 3.2 | Long-tail: cars >> bikes >> trains. 50% occluded |
| 3. Drivable Area | 3.4 | Direct vs. alternative — context beyond lane lines |
| 4. Lane Marking | 3.3 | 8 categories + direction + continuity. 100x more data than prior work |
| 5. Semantic Seg | 3.5 | 40 classes. Cityscapes model fails on BDD100K (domain gap) |
| 6. Multitask | Sec. 5 | Joint training improves harder tasks (seg, tracking) |
| 7. MOT | 3.6 | 49K occlusion events. 10x more sequences than MOT17 |
| 8. MOTS | 3.7 | Cascaded: instance seg → MOTS improves both |

**Next steps:**
- Fine-tune YOLOv8 specifically on BDD100K detection annotations
- Fine-tune SegFormer on BDD100K semantic segmentation split (10K images)
- Build a single-backbone multitask model (Tasks 2+3+5 simultaneously)
- Study the domain adaptation experiments (city vs. highway, day vs. night)